# Step 1: Problem Framing
## Horizon-Aware Startup Outcome Prediction

## 1.1 Problem Definition

**Core scientific question:** Among startups that reached a terminal outcome in this Crunchbase dataset, how much of outcome prediction is real early signal versus future-information leakage embedded in funding-history columns?

**Dataset:** The Crunchbase investments dataset (`investments_VC 2.csv`) contains 49,438 startup records with 39 raw columns covering identity, geography, founding dates, funding rounds, funding amounts, and outcome status.

**Target variable:** Binary classification of terminal startup outcome:
- `acquired` = 1 (positive class) — the startup was acquired
- `closed` = 0 (negative class) — the startup shut down

**Working dataset:** After excluding right-censored `operating` firms (see §1.2), the terminal-outcome subset contains approximately 6,295 startups (3,692 acquired / 2,603 closed) — a 59:41 split. This is mildly imbalanced but tractable without heavy-handed oversampling.

**The differentiating insight:** This is not "fancier model wins." It is "validity-aware modelling changes the answer." The dataset's funding columns (`seed`, `venture`, `round_A`–`round_H`, `funding_total_usd`) are **lifetime aggregates**, not point-in-time snapshots. Using them as-is contaminates the prediction with information that accumulated during and after the outcome process. The three-horizon design (§1.3) turns a routine classification project into a methodological contribution by explicitly measuring this contamination.

## 1.2 Censoring Argument — Excluding 'Operating' Startups

The raw dataset contains three status values: `operating` (~41,829), `acquired` (~3,692), and `closed` (~2,603).

**Why 'operating' must be excluded:**

1. **Right-censoring:** Operating startups have not yet reached a terminal outcome. They may eventually be acquired, close, or IPO — we simply don't know yet. Including them as a third class falsely treats "not yet resolved" as a distinct outcome category.

2. **Survivorship bias risk:** Treating operating firms as implicit negatives (i.e., "not acquired") would contaminate the closed class with healthy companies that haven't failed — they just haven't exited yet. This systematically biases the model toward associating "operating" characteristics with failure.

3. **Class contamination:** The operating class is heterogeneous by definition — it contains firms at every stage of their lifecycle, from newly founded to mature. Because their outcomes are unobserved, any assignment to acquired or closed is speculation, not data. <!-- REPORT NOTE: Tighten this point for the report — the original draft made a speculative claim about specific sub-populations (pre-acquisition targets, pre-failure firms) that cannot be verified from the data. Keep the argument grounded in what we can observe: outcomes are missing, not latent. -->

4. **Statistical justification:** In survival analysis terms, operating firms are right-censored observations. Standard classification models have no mechanism to handle censoring — they require observed outcomes. The correct approach is to exclude censored observations and model only the resolved population.

**Consequence:** The working dataset shrinks from ~49,438 to ~6,295 records. This is a substantial reduction but methodologically necessary. The alternative — including operating firms — would inflate sample size at the cost of target validity, which is a poor trade-off for a project centred on predictive integrity.

## 1.3 The Three Prediction Horizons

The central methodological design of this project is a three-horizon framework that controls for temporal information leakage. The raw Crunchbase data presents funding columns as lifetime aggregates — total amounts raised across all rounds up to the snapshot date. These are **not** point-in-time features; they encode information that only becomes available after the startup has progressed (or failed) through its funding lifecycle.

By restricting features to what was genuinely observable at different points in time, we can measure how much apparent predictive power is real versus how much is an artefact of future information leaking into the feature set.

| Horizon | Code Name | Features Allowed | Purpose |
|---------|-----------|------------------|---------|
| **H1: Founding-time** | `founding_dataset` | Sector/category, market, geography, founding year/quarter | **Most defensible model** — only uses information available at the moment a company is founded. Answers the cleanest causal question: can we predict outcome from initial conditions alone? |
| **H2: First-funding** | `first_funding_dataset` | H1 features + first funding date, time-to-first-funding lag | **Practical compromise** — adds a small amount of temporal information (when did funding first arrive?) that is still defensible and commonly available to early-stage investors. |
| **H3: Full snapshot** | `snapshot_dataset` | All funding aggregates, all round amounts, total funding | **Upper bound / leakage demonstration** — shows how much performance inflates when we allow future information. High AUC here doesn't mean the model is good; it means the features leak the outcome. |

**Main narrative framing:** The story is about the *tension between validity and utility*. H1 is the most honest model. H3 shows what you get if you don't care about leakage. H2 is the middle ground. The **signature finding** of this project is the AUC gap between H1 and H3 — the gap quantifies how much performance is real signal versus temporal contamination.

## 1.4 Metric Definitions

Given the binary classification setup (acquired vs closed) with a 59:41 class split, we use a suite of complementary metrics. No single metric tells the full story.

| Metric | Role | Why This Metric |
|--------|------|-----------------|
| **ROC-AUC** (primary) | Threshold-independent discrimination | Measures the model's ability to rank acquired firms above closed firms across all classification thresholds. Robust to mild class imbalance. The primary metric for cross-horizon and cross-model comparison. |
| **PR-AUC** | Positive-class-focused discrimination | More sensitive than ROC-AUC when positive-class recall matters. If the dataset were more imbalanced, this would be the primary metric. Used as a robustness check. |
| **Balanced Accuracy** | Intuitive summary at chosen threshold | Average of sensitivity and specificity. Easy to interpret and communicate. Computed at the default 0.5 threshold and at the threshold that maximises Youden's J statistic. |
| **F1** | Harmonic mean of precision and recall | Penalises models that sacrifice one for the other. Computed at the threshold chosen for balanced accuracy. |
| **Brier Score** | Calibration quality | Measures the mean squared difference between predicted probabilities and actual outcomes. Lower is better. Critical for assessing whether predicted probabilities are reliable for decision-making. |
| **ECE (Expected Calibration Error)** | Predicted probability reliability | Bins predictions and measures the average absolute gap between predicted confidence and observed frequency. Complements the Brier score with a more granular view of calibration. |

**Threshold selection:** For threshold-dependent metrics (balanced accuracy, F1), we will select the threshold on the validation set using Youden's J statistic (maximises sensitivity + specificity - 1), then apply that fixed threshold to the test set.

## 1.5 Hard Constraints

Five non-negotiable constraints govern the entire project pipeline:

1. **Leakage control** — Enforced programmatically by column registries in `src/config.py`. Every column in the raw CSV is classified by temporal availability (`safe_at: founding | first_funding | snapshot | never`). Automated tests in `tests/test_preprocessing.py` assert that no column from a later horizon appears in an earlier horizon's feature matrix. No funding aggregate column may appear in H1 or H2 features.

2. **Temporal validation** — The train/val/test split is chronological by `first_funding_year`, not random. Train: ≤2008, Validation: 2009–2010, Test: ≥2011. This reflects real-world deployment (predicting future outcomes from past data) and tests for temporal drift. No stratified random splitting, even if it would produce "better" metrics.

3. **Reproducibility** — All random seeds are fixed (42 throughout). The full pipeline is scripted in notebooks that run end-to-end. All dependencies are pinned in `requirements.txt`. Data processing is deterministic — no operations depend on system state or randomness.

4. **Interpretability** — SHAP explanations are computed for the best-performing models on each horizon. Feature importance is not just a leaderboard — it reveals whether the model is using genuine founding-time signals (H1) or is effectively "reading the answer" through funding aggregates (H3).

5. **Laptop-scale** — All computation must complete on a 32GB Intel MacBook without GPU. This constrains model choices (TabM must run on CPU) and hyperparameter search budgets (Optuna trials capped). If TabM proves infeasible, a standard PyTorch feedforward network with entity embeddings serves as the fallback.

## 1.6 Leak Registry

The leak registry in `src/config.py` classifies all 39 raw columns in the dataset. Each column is assigned:
- **`safe_at`**: the earliest horizon where the column's value is genuinely observable (`founding`, `first_funding`, `snapshot`, or `never`)
- **`risk_notes`**: a human-readable explanation of why the column is or isn't safe at earlier horizons
- **`decision`**: one of five actions — `include` (direct feature), `include_h3_only` (snapshot horizon only), `exclude` (never used), `derive` (used to engineer features then dropped), `target` (the prediction target)

This registry is the single source of truth for horizon enforcement. The automated tests assert that the feature matrices for each horizon dataset contain only columns that are `safe_at` that horizon level or earlier.

In [ ]:
import sys
sys.path.insert(0, '..')
from src.config import LEAK_REGISTRY, FORBIDDEN, FOUNDING_SAFE, FIRST_FUNDING_SAFE, SNAPSHOT_ALL
import pandas as pd

# Display the full leak registry as a table
leak_df = pd.DataFrame(LEAK_REGISTRY).T
leak_df.index.name = 'column'
print(f"Total columns classified: {len(leak_df)}")
print(f"  - Excluded (identifiers/post-outcome): {(leak_df['decision'] == 'exclude').sum()}")
print(f"  - Target:                               {(leak_df['decision'] == 'target').sum()}")
print(f"  - Derive (date → features, then drop):  {(leak_df['decision'] == 'derive').sum()}")
print(f"  - Include (founding-safe):               {((leak_df['decision'] == 'include') & (leak_df['safe_at'] == 'founding')).sum()}")
print(f"  - Include H3 only (snapshot):            {(leak_df['decision'] == 'include_h3_only').sum()}")
print()

# Verify horizon lists are consistent with registry
founding_in_registry = [c for c in FOUNDING_SAFE if c in LEAK_REGISTRY]
snapshot_only = [c for c in SNAPSHOT_ALL if c not in FIRST_FUNDING_SAFE]
print(f"FOUNDING_SAFE columns: {len(FOUNDING_SAFE)}")
print(f"FIRST_FUNDING_SAFE adds: {len(FIRST_FUNDING_SAFE) - len(FOUNDING_SAFE)} column(s)")
print(f"SNAPSHOT_ALL adds: {len(snapshot_only)} columns beyond first-funding")
print(f"FORBIDDEN columns: {len(FORBIDDEN)}")

leak_df

## 1.7 Agent Governance Plan

This project uses Claude Code (Claude Opus) as an AI coding agent throughout. The agent governance plan defines clear boundaries for what is delegated versus independently verified.

### Delegated to the agent:
- **Code generation:** Pipeline scaffolding, data loading, cleaning functions, model training loops, plotting utilities
- **Boilerplate:** Notebook structure, docstrings, test scaffolds, config file structure
- **Initial model configurations:** Default hyperparameter ranges, Optuna search spaces, early stopping settings
- **Plot formatting:** Axis labels, colour schemes, figure sizing, SHAP plot generation

### Verified independently by me:
- **Leakage classifications:** Every column's `safe_at` assignment in the leak registry was manually reviewed against the data dictionary and the temporal logic of each horizon
- **Horizon boundary enforcement:** Assertions that no snapshot-only column appears in H1 or H2 feature matrices
- **Target variable definition:** The decision to exclude operating firms and the binary acquired/closed encoding
- **Temporal split boundaries:** The chronological train/val/test split and its rationale
- **Metric interpretation:** Whether AUC differences are meaningful, whether calibration matters for the use case
- **Final model selection:** The chosen model for each horizon, and the narrative about what the AUC gap means
- **Report narrative:** All analytical claims, framing choices, and conclusions

### Logging protocol:
Every agent contribution is logged in `logs/WORKFLOW_LOG.md` with a corresponding entry in `logs/DECISION_REGISTER.md`. Each entry records: what the agent proposed, what risk type was involved, whether the contribution was accepted/modified/rejected, and what verification was performed. The Decision Register becomes the appendix directly.

## 1.8 Model Stack

Five models are compared across horizons. The stack is designed to span the complexity spectrum from trivial baselines to modern deep learning, with each model serving a specific analytical role.

| Model | Role | Why This Model |
|-------|------|----------------|
| **DummyClassifier** | Sanity check | Establishes the random-performance baseline. If any real model fails to beat this, something is fundamentally wrong. Uses `strategy="most_frequent"` and `strategy="stratified"`. Expected AUC ~0.50. |
| **LogisticRegression** | Interpretable statistical baseline | The cleanest interpretable benchmark. Coefficients are directly comparable across horizons. Provides a credible performance floor. If the problem has any linear signal, LR will find it. |
| **HistGradientBoostingClassifier** | Nonlinear sklearn-native baseline | Handles missing values natively (no imputation needed), no external dependencies, fast training. Captures nonlinear feature interactions without the complexity of external libraries. |
| **CatBoostClassifier** | Industry-grade gradient-boosted trees | Native categorical handling is critical for our high-cardinality `market` and `category_list` fields. Handles missingness natively. Well-documented, reproducible, and the likely best-performing traditional ML model. |
| **TabM** | Modern tabular deep learning | ICLR 2025 paper — current state-of-the-art for tabular deep learning. Pip-installable with a clean API. Represents the neural network component of the model stack. If it fails on CPU, falls back to a standard PyTorch feedforward NN with entity embeddings. |

**Horizon coverage:**
- **H1 (founding) and H2 (first-funding):** Full stack — all 5 models. This is where the valid comparison lives.
- **H3 (snapshot):** LogisticRegression + CatBoost only. Enough to demonstrate the leakage-driven AUC inflation without over-investing compute on a deliberately contaminated feature set.

## 1.9 Validation Strategy

### Temporal Split (Primary)

The dataset is split chronologically by `first_funding_year` for firms with terminal outcomes:

| Split | Years | Purpose |
|-------|-------|---------|
| **Train** | ≤ 2008 | Model fitting |
| **Validation** | 2009–2010 | Hyperparameter tuning, threshold selection, model comparison |
| **Test** | ≥ 2011 | Final evaluation only — opened once |

This reflects real-world deployment: we train on historical data and predict future outcomes. It also tests for temporal drift — if startup characteristics or acquisition patterns changed over time, the temporal split will reveal it.

**Why not random stratified split?** A random split allows future data points to leak into the training set. A model trained on 2005–2013 data (randomly sampled) could memorise era-specific patterns that wouldn't generalise to new cohorts. The temporal split is stricter but more honest.

### Tuning Protocol
- **LogisticRegression and HGB:** 5-fold stratified CV within the training set
- **CatBoost and TabM:** Validation-based early stopping using the held-out validation set
- **Hyperparameter search:** Optuna with budget limits (20–50 trials depending on model complexity)

## 1.10 Summary

This notebook defines the complete problem framing before any data exploration or modelling begins:

1. **Target:** Binary classification — acquired (1) vs closed (0), with operating firms excluded as right-censored
2. **Horizons:** Three feature sets (H1 founding, H2 first-funding, H3 snapshot) to measure leakage inflation
3. **Metrics:** ROC-AUC primary, supplemented by PR-AUC, balanced accuracy, F1, Brier score, and ECE
4. **Constraints:** Leakage control via column registries, temporal validation, reproducibility, interpretability, laptop-scale
5. **Models:** DummyClassifier → LogisticRegression → HGB → CatBoost → TabM (full stack on H1/H2, reduced on H3)
6. **Validation:** Chronological split by first_funding_year (train ≤2008, val 2009–2010, test ≥2011)
7. **Agent governance:** Clear delegation boundaries with mandatory per-decision logging

The next step (notebook 02) conducts the exploratory data analysis and data audit.